# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

For each record set, we display its `@id`, name, description, and list its fields by their `@id`s and names.

In [ ]:
# List available record sets and their fields by their @id
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets found in the Croissant schema.')
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[no name]')}")
        print(f"  Description: {rs.get('description', '[no description]')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('  Fields:')
        for f in fields:
            if isinstance(f, dict):
                print(f"    - @id: {f['@id']}  Name: {f.get('name', '[no name]')}")
            else:
                print(f"    - @id: {f}")
        print('-' * 50)

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrame(s) for analysis. Use the record set and field `@id`s from the overview.

`mlcroissant` allows loading each record set via its `@id`. We'll extract each record set and display the columns (field `@id`s) for inspection.

In [ ]:
# Prepare a list of record set @id values from the overview
# Example placeholder, replace with discovered @ids if available (update as needed based on actual dataset content):
record_set_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for Record Set {record_set_id}: {list(df.columns)}")
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

# Select one record set for EDA below
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"Using {example_record_set_id} for subsequent analysis.")
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize fields, categorize/group data.

We will select a numeric field by its `@id` and perform filtering, normalization, and grouping operations. Please update field `@id` placeholders below with actual column names if known.

In [ ]:
import numpy as np

if example_record_set_id is not None:
    df = dataframes[example_record_set_id]
    # Find a numeric field (column) for this demonstration. Replace with field @id as appropriate.
    candidate_numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
        print(f"Selected numeric field: {numeric_field_id}")
        # Filter where value exceeds arbitrary threshold (adjust as needed)
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (
                filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Attempt to group by another field (if available)
        group_field_candidates = [col for col in df.columns if col != numeric_field_id]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            grouped_df.columns = [f"mean_{numeric_field_id}"]
            print(f"Grouped data by {group_field} (means of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print('No numeric fields found for EDA.')
else:
    print('No dataframe loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships in the selected record set. We plot the distribution of the selected numeric field and, if available, a grouped barplot of means by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting
if example_record_set_id is not None and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Grouped means barplot (if applicable)
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        grouped_mean = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped_mean, x=group_field, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('Data not available for visualization.')

## 6. Conclusion
In this notebook, we have demonstrated how to use the `mlcroissant` library to load, inspect, and analyze a Croissant-described dataset. We performed data overview, extraction, preliminary EDA including filtering and normalization, and visualized core aspects of a selected record set. 

For more advanced analysis, update record set and field `@id` references as needed once the full schema structure is known.